# SmolVLA Multitask Training: Pushing + Insertion

One shared model trained simultaneously on two tasks via `ConcatDataset` + random shuffle.

**Task pair (verified compatible):**
- `ganker5/so100_push_20250331` — v2.1, cameras: laptop + phone, action=[6]
- `samsam0510/tape_insert_1` — v2.0, cameras: laptop + phone, action=[6]

**Why not pick_and_place:** `SeanLMH/so100_picknplace_v2` uses `front`/`overhead` cameras — incompatible with pushing's `laptop`/`phone`. Verified against HuggingFace `info.json`.

**Overnight run:** 5,000 steps ≈ 2 epochs over ~77k combined frames at batch_size=32.

In [ ]:
# install phiên bản LeRobot cũ hơn hỗ trợ v1.6/v2.0
# LeRobot 0.3.x hỗ trợ dataset format v1.6
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall",
                "lerobot", "-y", "-q"])

for version in ["0.3.3", "0.3.2", "0.3.1", "0.3.0"]:
    print(f"Trying lerobot=={version}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install",
         f"lerobot=={version}", "-q"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"Installed {version}")
        break
    else:
        print(f"  Failed: {result.stderr[:100]}")

In [ ]:
# ============================================================
# PATCH 1 — modeling_smolvla.py: attention mask shape fix
# ============================================================
import importlib
import lerobot.policies.smolvla.modeling_smolvla as _m

src_path = _m.__file__
with open(src_path) as f:
    src = f.read()

OLD = "    pad_2d_masks = pad_masks[:, None, :] * pad_masks[:, :, None]\n    att_2d_masks = att_2d_masks & pad_2d_masks"
NEW = """    if pad_masks.shape[1] < att_2d_masks.shape[-1]:
        import torch as _t
        _diff = att_2d_masks.shape[-1] - pad_masks.shape[1]
        _extra = _t.ones(pad_masks.shape[0], _diff,
                         device=pad_masks.device, dtype=pad_masks.dtype)
        pad_masks = _t.cat([_extra, pad_masks], dim=1)
    pad_2d_masks = pad_masks[:, None, :] * pad_masks[:, :, None]
    att_2d_masks = att_2d_masks & pad_2d_masks"""

if OLD in src:
    patched = src.replace(OLD, NEW)
    with open(src_path, "w") as f:
        f.write(patched)
    importlib.reload(_m)
    print("✅ Patched and reloaded:", src_path)
elif NEW in src:
    print("✅ Patch already applied")
else:
    print("⚠️ OLD pattern not found — check lines around 225:")
    for i, line in enumerate(src.splitlines()[220:230], start=221):
        print(f"  {i}: {repr(line)}")

In [ ]:
# Install system FFmpeg (required for video decoding in DataLoader)
import subprocess, sys

ffmpeg_result = subprocess.run(
    ["sudo", "apt-get", "install", "-y", "-q", "ffmpeg"],
    capture_output=True, text=True
)
if ffmpeg_result.returncode == 0:
    print("FFmpeg installed/confirmed")
else:
    print(f"apt install failed: {ffmpeg_result.stderr[:200]}")

subprocess.run([sys.executable, "-m", "pip", "install", "num2words", "-q"], check=False)
print("num2words ready")

In [ ]:
# ============================================================
# PATCH 2 — embed_prefix: drop phantom tail entries
# ============================================================
import torch

import lerobot.policies.smolvla.modeling_smolvla as _smolvla
from lerobot.policies.smolvla.modeling_smolvla import VLAFlowMatching

if not hasattr(_smolvla, "_patched_embed_prefix_done"):
    _orig_embed_prefix = VLAFlowMatching.embed_prefix

    def _patched_embed_prefix(self, images, img_masks, lang_tokens, lang_masks, state=None):
        embs, pad_masks, att_masks = _orig_embed_prefix(
            self, images, img_masks, lang_tokens, lang_masks, state=state
        )
        n_emb = embs.shape[1]
        if att_masks.shape[1] > n_emb:
            att_masks = att_masks[:, :n_emb]
        return embs, pad_masks, att_masks

    VLAFlowMatching.embed_prefix = _patched_embed_prefix
    _smolvla._patched_embed_prefix_done = True
    print("✅ embed_prefix patched")
else:
    print("✅ embed_prefix patch already applied")

In [ ]:
# ============================================================
# PATCH 3 — smolvlm_with_expert.py: expert token attention mask
# ============================================================
import importlib
import lerobot.policies.smolvla.smolvlm_with_expert as _se

src_path = _se.__file__
with open(src_path) as f:
    lines = f.readlines()

KEY = "torch.where(attention_mask[:, None, :, :], att_weights, big_neg)"
patched = False

for i, line in enumerate(lines):
    if KEY in line and "shape[-1] < att_weights" not in line:
        ind = len(line) - len(line.lstrip())
        p = " " * ind
        lines[i] = (
            f"{p}# Fix: pad attention_mask when smaller than att_weights\n"
            f"{p}if attention_mask.shape[-1] < att_weights.shape[-1]:\n"
            f"{p}    import torch as _t\n"
            f"{p}    _orig_q = attention_mask.shape[-2]\n"
            f"{p}    _orig_k = attention_mask.shape[-1]\n"
            f"{p}    _q_len = att_weights.shape[-2]\n"
            f"{p}    _k_len = att_weights.shape[-1]\n"
            f"{p}    _exp = _t.zeros((attention_mask.shape[0], _q_len, _k_len), dtype=attention_mask.dtype, device=attention_mask.device)\n"
            f"{p}    _exp[:, :_orig_q, :_orig_k] = attention_mask\n"
            f"{p}    if _q_len > _orig_q:\n"
            f"{p}        _exp[:, _orig_q:, :] = True\n"
            f"{p}    attention_mask = _exp\n"
            f"{line}\n"
        )
        patched = True
        print(f"✅ Patched line {i+1} in {src_path}")
        break

if patched:
    with open(src_path, "w") as f:
        f.writelines(lines)
    importlib.reload(_se)
    print("✅ Saved and reloaded smolvlm_with_expert")
else:
    print("✅ Patch already applied or pattern not found")

In [ ]:
# ============================================================
# CONFIG
# ============================================================
import os, json

os.environ["HF_TOKEN"] = ""

TASKS = {
    "pushing":   "ganker5/so100_push_20250331",
    "insertion": "samsam0510/tape_insert_1",
}
SPLITS_PATH  = "splits.json"   # reuse existing 80/20 splits
OUTPUT_DIR   = os.path.expanduser("~/smolvla_mini/multitask")
BATCH_SIZE   = 32
TOTAL_STEPS  = 5_000           # ~2 epochs over ~77k combined frames
LOG_FREQ     = 50
SAVE_FREQ    = 1_000           # 5 checkpoints total
SEED         = 42
NUM_WORKERS  = 4               # drop to 0 if video decode errors appear
DEVICE       = "cuda"

print(f"Output dir : {OUTPUT_DIR}")
print(f"Tasks      : {list(TASKS.keys())}")
print(f"Steps      : {TOTAL_STEPS}")

In [ ]:
# ============================================================
# DATASET LOADING & MERGING
# Core change vs single-task: ConcatDataset from two datasets
# Force pyav backend to avoid TorchCodec/libtorchcodec loading
# ============================================================
import torch
from torch.utils.data import ConcatDataset, DataLoader
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.datasets.compute_stats import aggregate_stats

with open(SPLITS_PATH) as f:
    splits = json.load(f)

# Load each dataset filtered to train episodes only.
# episodes= arg computes filtered stats automatically — no Subset needed.
# Force pyav so data loading never touches torchcodec in this environment.
ds_push = LeRobotDataset(
    TASKS["pushing"],
    episodes=splits["pushing"]["train_episodes"],
    video_backend="pyav",
)
ds_ins = LeRobotDataset(
    TASKS["insertion"],
    episodes=splits["insertion"]["train_episodes"],
    video_backend="pyav",
)

# Verify schema compatibility (checked offline via info.json, assert here as a guard)
assert set(ds_push.meta.camera_keys) == set(ds_ins.meta.camera_keys), (
    f"Camera key mismatch: {ds_push.meta.camera_keys} vs {ds_ins.meta.camera_keys}"
)
assert ds_push.meta.shapes["action"] == ds_ins.meta.shapes["action"], (
    f"Action shape mismatch: {ds_push.meta.shapes['action']} vs {ds_ins.meta.shapes['action']}"
)

# Simple concatenation — DataLoader shuffle=True gives random interleaving
combined_train = ConcatDataset([ds_push, ds_ins])

print(f"Pushing   train frames: {len(ds_push)}")
print(f"Insertion train frames: {len(ds_ins)}")
print(f"Combined  total frames: {len(combined_train)}")
print(f"Camera keys: {sorted(ds_push.meta.camera_keys)}")

train_loader = DataLoader(
    combined_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)


In [ ]:
# ============================================================
# POLICY WITH AGGREGATED STATS
# aggregate_stats() computes count-weighted mean + pooled variance
# across both datasets — required for correct input normalization.
# ============================================================
import copy
import numpy as np
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig

def _inject_count(stats, n_frames):
    """Older dataset formats (v2.0/v2.1) omit 'count'; inject shape-(1,) count."""
    stats = copy.deepcopy(stats)
    for feat_stats in stats.values():
        if "count" not in feat_stats:
            feat_stats["count"] = np.array([n_frames])
    return stats

push_stats = _inject_count(ds_push.meta.stats, len(ds_push))
ins_stats  = _inject_count(ds_ins.meta.stats,  len(ds_ins))
combined_stats = aggregate_stats([push_stats, ins_stats])

# Sanity check: aggregate_stats returns numpy arrays
import torch
action_mean = combined_stats["action"]["mean"]
assert np.isfinite(action_mean).all(), f"Non-finite action mean: {action_mean}"
print(f"Action mean (combined): {action_mean.tolist()}")

# Feature definitions — identical for both datasets
features        = dataset_to_policy_features(ds_push.meta.features)
output_features = {k: v for k, v in features.items() if v.type.name == "ACTION"}
input_features  = {k: v for k, v in features.items() if k not in output_features}

cfg = SmolVLAConfig(
    chunk_size=50,
    num_vlm_layers=16,
    expert_width_multiplier=0.75,
    train_expert_only=True,
    load_vlm_weights=False,
    pad_language_to="longest",
    use_cache=False,
)
cfg.input_features  = input_features
cfg.output_features = output_features

policy = SmolVLAPolicy(config=cfg, dataset_stats=combined_stats)
policy.to(DEVICE)
policy.train()

n_trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"Trainable params: {n_trainable:,}")

In [ ]:
# ============================================================
# OPTIMIZER & SCHEDULER (same hyperparams as single-task)
# ============================================================
from lerobot.optim.optimizers import AdamWConfig
from lerobot.optim.schedulers import CosineDecayWithWarmupSchedulerConfig

# get_optim_params() respects train_expert_only=True — only trains expert layers
optimizer = AdamWConfig(
    lr=1e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=1e-10,
    grad_clip_norm=10,
).build(policy.get_optim_params())

scheduler = CosineDecayWithWarmupSchedulerConfig(
    num_warmup_steps=100,
    num_decay_steps=4_900,   # TOTAL_STEPS - warmup = full cosine decay over run
    peak_lr=1e-4,
    decay_lr=2.5e-6,
).build(optimizer, num_training_steps=TOTAL_STEPS)

print("Optimizer and scheduler ready")

In [ ]:
# ============================================================
# WANDB
# ============================================================
import wandb
wandb.login("")
wandb.init(
    project="smolvla_mini",
    name="multitask_push_insert",
    config={
        "tasks":            list(TASKS.keys()),
        "total_steps":      TOTAL_STEPS,
        "batch_size":       BATCH_SIZE,
        "combined_frames":  len(combined_train),
        "lr":               1e-4,
        "num_vlm_layers":   16,
        "chunk_size":       50,
        "expert_mult":      0.75,
    }
)

In [ ]:
# ============================================================
# RUNTIME PATCH — fix attention_mask/att_weights shape mismatch
# Root cause: att_2d_masks is built from pad_masks (e.g. 193 tokens)
# but prefix_embs+suffix_embs is shorter (e.g. 175 tokens), so
# att_weights[seq_k=175] ≠ attention_mask[seq_k=193].
# This monkey-patch trims/pads the mask to match key_states directly.
# More reliable than file patching — no anchor text search needed.
# ============================================================
import torch
from lerobot.policies.smolvla.smolvlm_with_expert import SmolVLMWithExpertModel

if not hasattr(SmolVLMWithExpertModel, "_eager_attn_patched"):
    _orig_eager = SmolVLMWithExpertModel.eager_attention_forward

    def _patched_eager(self, attention_mask, batch_size, head_dim,
                       query_states, key_states, value_states):
        seq_k = key_states.shape[1]
        seq_q = query_states.shape[1]
        # Fix last dim (key sequence length)
        if attention_mask.shape[-1] > seq_k:
            attention_mask = attention_mask[..., :seq_k]
        elif attention_mask.shape[-1] < seq_k:
            pad = torch.ones(*attention_mask.shape[:-1], seq_k - attention_mask.shape[-1],
                             dtype=attention_mask.dtype, device=attention_mask.device)
            attention_mask = torch.cat([attention_mask, pad], dim=-1)
        # Fix second-to-last dim (query sequence length)
        if attention_mask.shape[-2] > seq_q:
            attention_mask = attention_mask[..., :seq_q, :]
        elif attention_mask.shape[-2] < seq_q:
            pad = torch.ones(*attention_mask.shape[:-2], seq_q - attention_mask.shape[-2], seq_k,
                             dtype=attention_mask.dtype, device=attention_mask.device)
            attention_mask = torch.cat([attention_mask, pad], dim=-2)
        return _orig_eager(self, attention_mask, batch_size, head_dim,
                           query_states, key_states, value_states)

    SmolVLMWithExpertModel.eager_attention_forward = _patched_eager
    SmolVLMWithExpertModel._eager_attn_patched = True
    print("✅ eager_attention_forward patched (runtime monkey-patch)")
else:
    print("✅ eager_attention_forward already patched")

In [ ]:
from lerobot.policies.smolvla.smolvlm_with_expert import SmolVLMWithExpertModel

def patched_attention(self, attention_mask, batch_size, head_dim,
                     query_states, key_states, value_states):

    attn_weights = torch.matmul(query_states, key_states.transpose(-2, -1))

    seq_k = key_states.shape[-2]

    # 🔥 FIX: force mask to match actual sequence
    if attention_mask.shape[-1] != seq_k:
        if attention_mask.shape[-1] > seq_k:
            attention_mask = attention_mask[..., :seq_k, :seq_k]
        else:
            pad = seq_k - attention_mask.shape[-1]
            attention_mask = torch.nn.functional.pad(
                attention_mask, (0, pad, 0, pad), value=False
            )

    big_neg = torch.finfo(attn_weights.dtype).min
    attn_weights = torch.where(attention_mask[:, None, :, :], attn_weights, big_neg)

    attn_probs = torch.softmax(attn_weights, dim=-1)
    return torch.matmul(attn_probs, value_states)

# apply patch
SmolVLMWithExpertModel.eager_attention_forward = patched_attention

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================
import time, torch, types
import torch.nn as nn
from pathlib import Path
from lerobot.datasets.utils import cycle
from lerobot.policies.smolvla.smolvlm_with_expert import SmolVLMWithExpertModel

# ── Self-contained replacement for eager_attention_forward ───────────────────
# Includes the mask-alignment fix inline — does NOT call _orig_eager, so the
# remote file's one-directional partial fix cannot interfere.
def _fixed_eager(self, attention_mask, batch_size, head_dim,
                 query_states, key_states, value_states):
    seq_k = key_states.shape[1]
    seq_q = query_states.shape[1]
    # Align attention_mask to (seq_q, seq_k) in both dimensions
    if attention_mask.shape[-1] > seq_k:
        attention_mask = attention_mask[..., :seq_k]
    elif attention_mask.shape[-1] < seq_k:
        pad = torch.ones(*attention_mask.shape[:-1], seq_k - attention_mask.shape[-1],
                         dtype=attention_mask.dtype, device=attention_mask.device)
        attention_mask = torch.cat([attention_mask, pad], dim=-1)
    if attention_mask.shape[-2] > seq_q:
        attention_mask = attention_mask[..., :seq_q, :]
    elif attention_mask.shape[-2] < seq_q:
        pad = torch.ones(*attention_mask.shape[:-2], seq_q - attention_mask.shape[-2], seq_k,
                         dtype=attention_mask.dtype, device=attention_mask.device)
        attention_mask = torch.cat([attention_mask, pad], dim=-2)
    # ── Original computation (reproduced in full, no call to self.method) ─────
    num_att_heads      = self.num_attention_heads
    num_kv_heads       = self.num_key_value_heads
    num_kv_groups      = num_att_heads // num_kv_heads
    seq_len            = key_states.shape[1]
    key_states   = key_states[:, :, :, None, :].expand(batch_size, seq_len, num_kv_heads, num_kv_groups, head_dim)
    key_states   = key_states.reshape(batch_size, seq_len, num_kv_heads * num_kv_groups, head_dim)
    value_states = value_states[:, :, :, None, :].expand(batch_size, seq_len, num_kv_heads, num_kv_groups, head_dim)
    value_states = value_states.reshape(batch_size, seq_len, num_kv_heads * num_kv_groups, head_dim)
    query_states = query_states.to(dtype=torch.float32).transpose(1, 2)
    key_states   = key_states.to(dtype=torch.float32).transpose(1, 2)
    att_weights  = torch.matmul(query_states, key_states.transpose(2, 3)) * (head_dim ** -0.5)
    att_weights  = att_weights.to(dtype=torch.float32)
    big_neg      = torch.finfo(att_weights.dtype).min
    att_weights  = torch.where(attention_mask[:, None, :, :], att_weights, big_neg)
    probs        = nn.functional.softmax(att_weights, dim=-1).to(dtype=value_states.dtype)
    att_output   = torch.matmul(probs, value_states.permute(0, 2, 1, 3))
    att_output   = att_output.permute(0, 2, 1, 3)
    att_output   = att_output.reshape(batch_size, -1, num_kv_heads * num_kv_groups * head_dim)
    return att_output

# ── Apply to BOTH the imported class AND the actual runtime class ─────────────
# If a prior reload created a second class object, type(vlm) catches it.
vlm = policy.model.vlm_with_expert
for cls in {SmolVLMWithExpertModel, type(vlm)}:
    cls.eager_attention_forward = _fixed_eager
# Belt-and-suspenders: also set on the instance dict so Python finds it first
vlm.eager_attention_forward = types.MethodType(_fixed_eager, vlm)
print(f"✅ eager_attention_forward patched on class={type(vlm).__name__} + instance")
# ─────────────────────────────────────────────────────────────────────────────

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

dl_iter    = cycle(train_loader)
loss_accum = 0.0
t0         = time.perf_counter()

policy.train()

for step in range(1, TOTAL_STEPS + 1):
    batch = next(dl_iter)
    batch = {
        k: v.to(DEVICE, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

    optimizer.zero_grad()
    try:
        loss, output_dict = policy(batch)
    except RuntimeError as e:
        msg = str(e)
        if "must match the size of tensor a" not in msg and "size of tensor a" not in msg:
            raise
        sample_losses = []
        B = next(v for v in batch.values() if isinstance(v, torch.Tensor)).shape[0]
        for i in range(B):
            single = {k: (v[i:i+1] if isinstance(v, torch.Tensor) else v)
                      for k, v in batch.items()}
            try:
                l, _ = policy(single)
                sample_losses.append(l)
            except Exception:
                continue
        if len(sample_losses) == 0:
            raise RuntimeError("All samples in batch failed during forward") from e
        loss = torch.stack(sample_losses).mean()
        output_dict = {}
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
    optimizer.step()
    scheduler.step()
    loss_accum += loss.item()

    if step % LOG_FREQ == 0:
        avg = loss_accum / LOG_FREQ
        lr  = optimizer.param_groups[0]["lr"]
        elapsed = (time.perf_counter() - t0) / 60
        print(f"Step {step:5d}/{TOTAL_STEPS} | loss={avg:.4f} | lr={lr:.2e} | {elapsed:.1f}min")
        wandb.log({"train/loss": avg, "train/lr": lr}, step=step)
        loss_accum = 0.0

    if step % SAVE_FREQ == 0 or step == TOTAL_STEPS:
        ckpt = output_path / "checkpoints" / f"step_{step:06d}" / "pretrained_model"
        ckpt.mkdir(parents=True, exist_ok=True)
        policy.save_pretrained(str(ckpt))
        last = output_path / "checkpoints" / "last" / "pretrained_model"
        last.mkdir(parents=True, exist_ok=True)
        policy.save_pretrained(str(last))
        print(f"  Checkpoint saved: step {step}")

wandb.finish()
print(f"\nTraining complete. Total time: {(time.perf_counter()-t0)/3600:.2f}h")

In [ ]:
# ============================================================
# EVALUATION — per-task test split loss
# Same pattern as training_singletask.ipynb cell 20
# Force pyav backend to avoid TorchCodec/libtorchcodec loading
# ============================================================
import numpy as np
from torch.utils.data import DataLoader, Subset
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

CKPT_PATH = str(output_path / "checkpoints" / "last" / "pretrained_model")


def get_frame_indices_for_episodes(dataset, episode_list):
    ep_set = set(episode_list)
    return [
        i for i, ep in enumerate(dataset.hf_dataset['episode_index'])
        if int(ep) in ep_set
    ]


def evaluate_task(task_name, repo_id, checkpoint_path, splits, device="cuda"):
    print(f"\n{'='*60}")
    print(f"  Evaluating: {task_name.upper()}")
    print(f"  Dataset   : {repo_id}")
    print(f"{'='*60}")

    dataset = LeRobotDataset(repo_id, video_backend="pyav")
    test_eps = splits[task_name]["test_episodes"]
    test_idx = get_frame_indices_for_episodes(dataset, test_eps)
    print(f"  Test episodes: {len(test_eps)} | Test frames: {len(test_idx)}")

    loader = DataLoader(
        Subset(dataset, test_idx),
        batch_size=16, shuffle=False, num_workers=0,
        pin_memory=False, drop_last=False,
    )

    policy = SmolVLAPolicy.from_pretrained(checkpoint_path)
    policy.eval().to(device)
    policy.config.empty_cameras = 0
    policy.config.pad_language_to = "max_length"
    policy.config.use_cache = False

    expected = set(policy.config.input_features.keys())
    losses = []
    skipped = 0

    for batch in loader:
        # Zero-pad any camera in config but missing from batch
        for key in expected:
            if 'images' in key and key not in batch:
                ref = next(k for k in batch if 'images' in k)
                batch[key] = torch.zeros_like(batch[ref])
        # Remove cameras not in config
        batch = {k: v for k, v in batch.items()
                 if 'images' not in k or k in expected}

        try:
            batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
            with torch.no_grad():
                _, info = policy(batch)
            losses.append(info["loss"])
        except Exception as e:
            print(f"  Skipping batch: {e}")
            skipped += 1

    result = {
        "task": task_name,
        "n_test_episodes": len(test_eps),
        "n_test_frames": len(test_idx),
        "n_evaluated": len(losses),
        "n_skipped": skipped,
        "mean_loss": float(np.mean(losses)),
        "std_loss": float(np.std(losses)),
        "min_loss": float(np.min(losses)),
        "max_loss": float(np.max(losses)),
    }
    print(f"  Mean loss: {result['mean_loss']:.4f} ± {result['std_loss']:.4f}")
    del policy, dataset
    torch.cuda.empty_cache()
    return result


mt_results = {}
for task, repo in TASKS.items():
    mt_results[task] = evaluate_task(task, repo, CKPT_PATH, splits)


In [ ]:
# ============================================================
# RESULTS COMPARISON: Singletask vs Multitask
# ============================================================

# Load singletask baselines (pushing=0.41, insertion=0.42)
ST_RESULTS_PATH = os.path.expanduser("~/evaluation_results.json")
st_results = {}
if os.path.exists(ST_RESULTS_PATH):
    with open(ST_RESULTS_PATH) as f:
        raw = json.load(f)
    if isinstance(raw, list):
        st_results = {r["category"]: r["mean_loss"] for r in raw if "category" in r}
    elif isinstance(raw, dict):
        st_results = {k: v.get("mean_loss", v.get("mean", float("nan")))
                      for k, v in raw.items()}

print(f"\n{'='*65}")
print("  MULTITASK EVALUATION — TEST SPLIT RESULTS")
print(f"{'='*65}")
print(f"{'Task':<15} {'Test Ep':>8} {'Test Fr':>8} {'Mean Loss':>11} {'Std':>8}")
print("-" * 55)
for task, res in mt_results.items():
    print(f"{task:<15} {res['n_test_episodes']:>8} {res['n_test_frames']:>8} "
          f"{res['mean_loss']:>11.4f} {res['std_loss']:>8.4f}")

if st_results:
    print(f"\n{'='*65}")
    print("  COMPARISON: Singletask vs Multitask")
    print(f"{'='*65}")
    print(f"{'Task':<15} {'Singletask':>12} {'Multitask':>12} {'Delta':>10}")
    print("-" * 53)
    for task in TASKS:
        st = st_results.get(task, float("nan"))
        mt = mt_results[task]["mean_loss"]
        delta = mt - st
        sign = "+" if delta >= 0 else ""
        print(f"{task:<15} {st:>12.4f} {mt:>12.4f} {sign}{delta:>9.4f}")

# Save results
results_path = str(output_path / "multitask_eval_results.json")
with open(results_path, "w") as f:
    json.dump(mt_results, f, indent=2)
print(f"\n✅ Results saved to: {results_path}")